<a href="https://colab.research.google.com/github/huifangsu6-cyber/BDAO_DSDO/blob/main/q8_opening_hours_alignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Q8 Python / pandas analysis
# Are restaurants open during peak-demand windows?

import pandas as pd
import numpy as np
import ast
import json
import re
from google.cloud import bigquery
from google.colab import auth

# ------------------------------------------------------------
# 1. Authenticate and connect to BigQuery
# ------------------------------------------------------------

auth.authenticate_user()

project_id = "bdao-group-yelp"
client = bigquery.Client(project=project_id)

# ------------------------------------------------------------
# 2. SQL 1: peak day-hour windows
# ------------------------------------------------------------

peak_windows_sql = """
WITH checkin_time AS (
  SELECT
    checkin_timestamp,

    EXTRACT(DAYOFWEEK FROM checkin_timestamp) AS day_of_week_number,

    CASE EXTRACT(DAYOFWEEK FROM checkin_timestamp)
      WHEN 1 THEN 'Sunday'
      WHEN 2 THEN 'Monday'
      WHEN 3 THEN 'Tuesday'
      WHEN 4 THEN 'Wednesday'
      WHEN 5 THEN 'Thursday'
      WHEN 6 THEN 'Friday'
      WHEN 7 THEN 'Saturday'
    END AS day_of_week_name,

    EXTRACT(HOUR FROM checkin_timestamp) AS checkin_hour

  FROM `bdao-group-yelp.yelp_dataset.yelp_checkin_clean`
  WHERE checkin_timestamp IS NOT NULL
),

day_hour_summary AS (
  SELECT
    day_of_week_number,
    day_of_week_name,
    checkin_hour,
    COUNT(*) AS checkin_count
  FROM checkin_time
  GROUP BY
    day_of_week_number,
    day_of_week_name,
    checkin_hour
),

total_checkins AS (
  SELECT
    SUM(checkin_count) AS total_checkin_count
  FROM day_hour_summary
),

ranked_peak_windows AS (
  SELECT
    d.day_of_week_number,
    d.day_of_week_name,
    d.checkin_hour,
    FORMAT('%02d:00-%02d:59', d.checkin_hour, d.checkin_hour) AS hour_block,
    d.checkin_count,
    ROUND(100 * d.checkin_count / t.total_checkin_count, 3) AS pct_of_all_checkins,
    RANK() OVER (ORDER BY d.checkin_count DESC) AS busiest_rank
  FROM day_hour_summary d
  CROSS JOIN total_checkins t
)

SELECT
  day_of_week_number,
  day_of_week_name,
  checkin_hour,
  hour_block,
  checkin_count,
  pct_of_all_checkins,
  busiest_rank
FROM ranked_peak_windows
ORDER BY busiest_rank
LIMIT 20
"""

df_peak = client.query(peak_windows_sql).to_dataframe()

print("Peak windows:")
display(df_peak)

# ------------------------------------------------------------
# 3. SQL 2: restaurant opening-hour data
# ------------------------------------------------------------

restaurants_sql = """
SELECT
  business_id,
  name,
  city,
  state,
  stars,
  is_open,
  review_count,
  categories,
  hours,
  RestaurantsPriceRange2,
  OutdoorSeating,
  Caters,
  RestaurantsReservations

FROM `bdao-group-yelp.yelp_dataset.yelp_restaurants`

WHERE stars IS NOT NULL
  AND hours IS NOT NULL
  AND categories IS NOT NULL
  AND REGEXP_CONTAINS(
    LOWER(categories),
    r'(^|,\\s*)restaurants(\\s*,|$)'
  )
"""

df_restaurants = client.query(restaurants_sql).to_dataframe()

print("Restaurant opening-hour rows loaded:", len(df_restaurants))
display(df_restaurants.head())

# ------------------------------------------------------------
# 4. Helper functions for parsing hours
# ------------------------------------------------------------

DAY_ORDER = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

NEXT_DAY = {
    "Monday": "Tuesday",
    "Tuesday": "Wednesday",
    "Wednesday": "Thursday",
    "Thursday": "Friday",
    "Friday": "Saturday",
    "Saturday": "Sunday",
    "Sunday": "Monday"
}

def parse_hours_value(value):
    """
    Safely parse the Yelp hours field.

    Handles:
    - Python dicts
    - dictionary-style strings with single quotes
    - JSON-style strings with double quotes
    - null / malformed values
    """

    if value is None:
        return {}

    # If BigQuery already returns a dictionary-like object
    if isinstance(value, dict):
        return value

    # Sometimes BigQuery can return nested rows or similar objects
    if hasattr(value, "items"):
        try:
            return dict(value)
        except Exception:
            return {}

    text = str(value).strip()

    if text == "" or text.lower() in ["none", "nan", "null"]:
        return {}

    # Try Python literal parsing first: {'Monday': '11:00-22:00'}
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, dict):
            return parsed
    except Exception:
        pass

    # Try JSON parsing: {"Monday": "11:00-22:00"}
    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict):
            return parsed
    except Exception:
        pass

    return {}


def time_to_minutes(time_text):
    """
    Convert a time string such as '11:00', '11:0', '0:0', or '23:30'
    into minutes after midnight.
    """

    if time_text is None:
        return None

    time_text = str(time_text).strip()

    match = re.match(r"^(\d{1,2}):(\d{1,2})$", time_text)
    if not match:
        return None

    hour = int(match.group(1))
    minute = int(match.group(2))

    # Handle occasional 24:00 representation
    if hour == 24 and minute == 0:
        return 24 * 60

    if hour < 0 or hour > 24 or minute < 0 or minute >= 60:
        return None

    return hour * 60 + minute


def parse_time_ranges(hours_string):
    """
    Parse one day's opening hours.

    Expected examples:
    - '11:00-22:00'
    - '17:00-02:00'
    - '0:0-0:0'

    If multiple ranges are separated by commas, this function tries to handle them.
    """

    if hours_string is None:
        return []

    text = str(hours_string).strip()

    if text == "" or text.lower() in ["none", "nan", "null", "closed"]:
        return []

    ranges = []

    # Most Yelp rows have one range, but this keeps the parser safer.
    for part in text.split(","):
        part = part.strip()

        if "-" not in part:
            continue

        start_text, end_text = part.split("-", 1)
        start_min = time_to_minutes(start_text)
        end_min = time_to_minutes(end_text)

        if start_min is None or end_min is None:
            continue

        ranges.append((start_min, end_min))

    return ranges


def hour_overlaps_interval(hour, start_min, end_min):
    """
    Check whether an hour block overlaps an opening interval.

    Example:
    hour = 18 means 18:00-18:59.
    """

    hour_start = hour * 60
    hour_end = (hour + 1) * 60

    return max(hour_start, start_min) < min(hour_end, end_min)


def get_open_day_hours(hours_value):
    """
    Convert a restaurant's hours dictionary into a set of open day-hour windows.

    Returns:
    set of tuples such as:
    {('Friday', 18), ('Friday', 19), ('Saturday', 1)}

    Handles overnight hours:
    Example:
    Friday 17:00-02:00 means:
    Friday 17:00-23:59 and Saturday 00:00-01:59.

    Note:
    A same start/end time such as 0:0-0:0 is treated as open 24 hours
    for that listed day.
    """

    hours_dict = parse_hours_value(hours_value)
    open_windows = set()

    for day in DAY_ORDER:
        if day not in hours_dict:
            continue

        ranges = parse_time_ranges(hours_dict.get(day))

        for start_min, end_min in ranges:

            # Case 1: same start and end time.
            # Yelp commonly uses 0:0-0:0 for 24-hour opening.
            if start_min == end_min:
                for hour in range(24):
                    open_windows.add((day, hour))
                continue

            # Case 2: normal same-day opening, e.g. 11:00-22:00
            if start_min < end_min:
                for hour in range(24):
                    if hour_overlaps_interval(hour, start_min, end_min):
                        open_windows.add((day, hour))

            # Case 3: overnight opening, e.g. 17:00-02:00
            else:
                # From start time to midnight on the listed day
                for hour in range(24):
                    if hour_overlaps_interval(hour, start_min, 24 * 60):
                        open_windows.add((day, hour))

                # From midnight to end time on the next day
                next_day = NEXT_DAY[day]
                for hour in range(24):
                    if hour_overlaps_interval(hour, 0, end_min):
                        open_windows.add((next_day, hour))

    return open_windows


# ------------------------------------------------------------
# 5. Build peak-window set from df_peak
# ------------------------------------------------------------

peak_windows = set(
    zip(
        df_peak["day_of_week_name"].astype(str),
        df_peak["checkin_hour"].astype(int)
    )
)

total_peak_windows = len(peak_windows)

print("Total peak windows:", total_peak_windows)
print("Peak windows:", sorted(peak_windows))

# ------------------------------------------------------------
# 6. Calculate peak-hour coverage for each restaurant
# ------------------------------------------------------------

def calculate_peak_coverage(hours_value):
    open_windows = get_open_day_hours(hours_value)

    if total_peak_windows == 0:
        return pd.Series({
            "peak_windows_covered": np.nan,
            "total_peak_windows": 0,
            "peak_coverage_rate": np.nan
        })

    covered = len(open_windows.intersection(peak_windows))
    coverage_rate = covered / total_peak_windows

    return pd.Series({
        "peak_windows_covered": covered,
        "total_peak_windows": total_peak_windows,
        "peak_coverage_rate": coverage_rate
    })


coverage_cols = df_restaurants["hours"].apply(calculate_peak_coverage)

df_q8 = pd.concat([df_restaurants.copy(), coverage_cols], axis=1)

# ------------------------------------------------------------
# 7. Create coverage groups
# ------------------------------------------------------------

def coverage_group(rate):
    if pd.isna(rate):
        return "Unknown"
    elif rate >= 0.75:
        return "High coverage"
    elif rate > 0:
        return "Partial coverage"
    else:
        return "No coverage"


df_q8["peak_coverage_group"] = df_q8["peak_coverage_rate"].apply(coverage_group)

# ------------------------------------------------------------
# 8. Final summary table required for the report
# ------------------------------------------------------------

summary = (
    df_q8
    .groupby("peak_coverage_group", dropna=False)
    .agg(
        restaurant_count=("business_id", "nunique"),
        avg_stars=("stars", "mean"),
        pct_4_stars_or_above=("stars", lambda x: (x >= 4.0).mean() * 100),
        avg_review_count=("review_count", "mean"),
        avg_peak_coverage_rate=("peak_coverage_rate", "mean")
    )
    .reset_index()
)

# Optional: sort groups in a logical order
group_order = {
    "High coverage": 1,
    "Partial coverage": 2,
    "No coverage": 3,
    "Unknown": 4
}

summary["sort_order"] = summary["peak_coverage_group"].map(group_order).fillna(99)

summary = (
    summary
    .sort_values("sort_order")
    .drop(columns="sort_order")
)

# Round for clean reporting
summary["avg_stars"] = summary["avg_stars"].round(3)
summary["pct_4_stars_or_above"] = summary["pct_4_stars_or_above"].round(2)
summary["avg_review_count"] = summary["avg_review_count"].round(1)
summary["avg_peak_coverage_rate"] = summary["avg_peak_coverage_rate"].round(3)

print("Q8 final output table:")
display(summary)

# ------------------------------------------------------------
# 9. Optional comparison: open vs closed restaurants by coverage group
# ------------------------------------------------------------

open_closed_summary = (
    df_q8
    .groupby(["peak_coverage_group", "is_open"], dropna=False)
    .agg(
        restaurant_count=("business_id", "nunique"),
        avg_stars=("stars", "mean"),
        pct_4_stars_or_above=("stars", lambda x: (x >= 4.0).mean() * 100),
        avg_review_count=("review_count", "mean"),
        avg_peak_coverage_rate=("peak_coverage_rate", "mean")
    )
    .reset_index()
)

open_closed_summary["avg_stars"] = open_closed_summary["avg_stars"].round(3)
open_closed_summary["pct_4_stars_or_above"] = open_closed_summary["pct_4_stars_or_above"].round(2)
open_closed_summary["avg_review_count"] = open_closed_summary["avg_review_count"].round(1)
open_closed_summary["avg_peak_coverage_rate"] = open_closed_summary["avg_peak_coverage_rate"].round(3)

print("Optional open vs closed comparison:")
display(open_closed_summary)

# ------------------------------------------------------------
# 10. Optional: inspect restaurants with no coverage
# ------------------------------------------------------------

no_coverage_preview = (
    df_q8[df_q8["peak_coverage_group"] == "No coverage"]
    [
        [
            "business_id",
            "name",
            "city",
            "state",
            "stars",
            "is_open",
            "review_count",
            "categories",
            "hours",
            "peak_windows_covered",
            "total_peak_windows",
            "peak_coverage_rate"
        ]
    ]
    .sort_values(["stars", "review_count"], ascending=[True, False])
    .head(20)
)

print("Optional preview: no-coverage restaurants")
display(no_coverage_preview)

Peak windows:


,day_of_week_number,day_of_week_name,checkin_hour,hour_block,checkin_count,pct_of_all_checkins,busiest_rank
0,1,Sunday,0,00:00-00:59,166963,1.961,1
1,7,Saturday,0,00:00-00:59,166465,1.955,2
2,7,Saturday,23,23:00-23:59,165307,1.941,3
3,6,Friday,23,23:00-23:59,156307,1.836,4
4,1,Sunday,1,01:00-01:59,144941,1.702,5
5,7,Saturday,1,01:00-01:59,144602,1.698,6
6,7,Saturday,22,22:00-22:59,143453,1.685,7
7,7,Saturday,18,18:00-18:59,143160,1.681,8
8,7,Saturday,17,17:00-17:59,139160,1.634,9
9,1,Sunday,17,17:00-17:59,135601,1.592,10


Restaurant opening-hour rows loaded: 44990


,business_id,name,city,state,stars,is_open,review_count,categories,hours,RestaurantsPriceRange2,OutdoorSeating,Caters,RestaurantsReservations
0,oQYZ3j2H758y07p4RpCWug,August Rhodes Bakery,Tucson,AZ,4.5,1,89,"Event Planning & Services, Food, Sandwiches, B...","{'Friday': '9:0-12:0', 'Monday': '0:0-0:0', 'S...",2.0,True,True,False
1,nFjk0xVI9fNiVN__5g-m8Q,Ichicoro Ane,Saint Petersburg,FL,4.0,1,264,"Ramen, Restaurants, Tapas/Small Plates, Nightl...","{'Friday': '15:0-2:0', 'Monday': '0:0-0:0', 'S...",2.0,False,True,True
2,Yh4OgbXQSpQ_ycK_FmNiGw,Suis Generis,New Orleans,LA,4.0,1,100,"American (New), Restaurants","{'Friday': '18:0-23:0', 'Monday': None, 'Satur...",2.0,True,False,False
3,hXBeTdtREKEMT2FMmzRPhQ,Steak & Pasta House,Reno,NV,4.0,0,59,"Steakhouses, Italian, Restaurants","{'Friday': '17:0-21:0', 'Monday': None, 'Satur...",2.0,False,None,True
4,jlgi2sLx5i6_t4mcODJW7A,Redemption,New Orleans,LA,3.0,0,63,"Restaurants, Breakfast & Brunch, Event Plannin...","{'Friday': '17:0-22:0', 'Monday': None, 'Satur...",3.0,True,None,True


Total peak windows: 20
Peak windows: [('Friday', 0), ('Friday', 22), ('Friday', 23), ('Saturday', 0), ('Saturday', 1), ('Saturday', 16), ('Saturday', 17), ('Saturday', 18), ('Saturday', 19), ('Saturday', 20), ('Saturday', 21), ('Saturday', 22), ('Saturday', 23), ('Sunday', 0), ('Sunday', 1), ('Sunday', 16), ('Sunday', 17), ('Sunday', 18), ('Sunday', 19), ('Thursday', 23)]
Q8 final output table:


,peak_coverage_group,restaurant_count,avg_stars,pct_4_stars_or_above,avg_review_count,avg_peak_coverage_rate
0,High coverage,7390,3.020,22.52,76.4,0.922
2,Partial coverage,32902,3.599,48.52,102.7,0.441
1,No coverage,4698,3.995,71.09,97.9,0.000


Optional open vs closed comparison:


,peak_coverage_group,is_open,restaurant_count,avg_stars,pct_4_stars_or_above,avg_review_count,avg_peak_coverage_rate
0,High coverage,0,2006,3.195,22.43,67.3,0.917
1,High coverage,1,5384,2.955,22.55,79.8,0.923
2,No coverage,0,1549,3.905,65.33,56.5,0.000
3,No coverage,1,3149,4.039,73.93,118.3,0.000
4,Partial coverage,0,9818,3.576,44.07,62.3,0.430
5,Partial coverage,1,23084,3.609,50.42,119.8,0.446


Optional preview: no-coverage restaurants


,business_id,name,city,state,stars,is_open,review_count,categories,hours,peak_windows_covered,total_peak_windows,peak_coverage_rate
38350,q8b9QZuB-s6mwr4E5MZUaw,Jack in the Box,Earth City,MO,1.0,1,16,"Restaurants, Fast Food, Burgers, Mexican, Brea...","{'Friday': '6:0-21:0', 'Monday': '6:0-21:0', '...",0.0,20.0,0.0
17408,Nu8aj_v77Fh3prSC_RoDWw,McDonald’s,Indianapolis,IN,1.0,1,6,"Coffee & Tea, Burgers, Food, Fast Food, Restau...","{'Friday': '5:0-11:0', 'Monday': '5:0-10:0', '...",0.0,20.0,0.0
19975,hAvgXT37uaQBfFmSjBPxEA,Cheer Counseling,Brandon,FL,1.0,0,5,"Professional Services, Health & Medical, Resta...","{'Friday': '9:0-20:0', 'Monday': '9:0-20:0', '...",0.0,20.0,0.0
43595,i03Bu6rZ1nqJ_5WTFBUGjA,IHOP,Nashville,TN,1.5,0,111,"Desserts, Breakfast & Brunch, American (New), ...","{'Friday': '7:0-15:0', 'Monday': '0:0-0:0', 'S...",0.0,20.0,0.0
37518,fsrRx1YBLp_opoczBE0bSQ,Steak ’n Shake,Indianapolis,IN,1.5,1,104,"Steakhouses, Breakfast & Brunch, Restaurants, ...","{'Friday': '14:0-17:0', 'Monday': '14:0-17:0',...",0.0,20.0,0.0
27403,cbqE6Y8OXvfhZuyz2Gdchw,Burger King,Reno,NV,1.5,1,65,"Fast Food, Restaurants, Burgers","{'Friday': '6:0-15:0', 'Monday': '6:0-15:0', '...",0.0,20.0,0.0
30033,mXUn9lx30yUGK0HRRBL0Rg,Burger King,Reno,NV,1.5,1,58,"Fast Food, Burgers, Restaurants","{'Friday': '7:0-14:0', 'Monday': '7:0-14:0', '...",0.0,20.0,0.0
39362,Cx3G-kw4kYtlcig8yG7fWw,Burger King,Sparks,NV,1.5,1,57,"Burgers, Fast Food, Restaurants","{'Friday': '7:0-15:0', 'Monday': '9:0-15:0', '...",0.0,20.0,0.0
10484,mF2v1J8RLSDPScs44wbYxQ,IHOP,Gallatin,TN,1.5,0,40,"Restaurants, American (Traditional), American ...","{'Friday': '7:0-15:0', 'Monday': '0:0-0:0', 'S...",0.0,20.0,0.0
17812,loqwRJxXq-jUQ40g75w6bQ,Manhattan Bagel,Langhorne,PA,1.5,1,31,"Sandwiches, Bagels, Food, Breakfast & Brunch, ...","{'Friday': '6:0-14:0', 'Monday': '6:0-14:0', '...",0.0,20.0,0.0


In [2]:
summary.to_csv('q8_summary_report.csv', index=False)
print('Summary table exported to q8_summary_report.csv')

Summary table exported to q8_summary_report.csv


The `summary` table has been exported to a CSV file named `q8_summary_report.csv`. You can download it from the Colab file browser (folder icon on the left panel).